## Этап 3.2 — Загрузка данных в SQLite: проверки качества и консистентности

### 1) Итоговая загрузка и базовая целостность
- Таблица `sessions`: **1,860,042** строк  
- Таблица `hits`: **15,726,470** строк  
- `DISTINCT session_id` в `hits`: **1,734,610** (то есть события покрывают ~93% сессий)
- Диапазоны дат совпадают:
  - `sessions.visit_date`: **2021-05-19 → 2021-12-31**
  - `hits.hit_date`: **2021-05-19 → 2021-12-31**

✅ Выгрузка в БД выглядит целостной: объемы ожидаемые, даты согласованы между таблицами.

---

### 2) “Сироты” (hits без соответствующей sessions)
Проверка `hits LEFT JOIN sessions` показала:
- `orphan_hits` (строк в `hits`, для которых нет `session_id` в `sessions`): **41,251**
- `orphan_sessions` (уникальных `session_id` среди сирот): **2,344**
- Доля сирот от общего числа hits: **0.2623%**

Наблюдение:
- У отдельных “сиротских” `session_id` может быть заметное число событий (в топе — сотни строк на одну сессию), но общий вклад в датасет небольшой.

📌 Интерпретация:
- Это небольшая, но реальная неконсистентность исходных данных (вероятно особенности выгрузки/склейки).  
- Для аналитики/ML корректнее **не пытаться “чинить”** это на уровне базы (не удалять автоматически), а осознанно учитывать в логике датасета (см. ниже “Правила использования”).

---

### 3) Сессии без событий (sessions без hits)
Проверка `sessions LEFT JOIN hits` показала:
- `sessions_without_hits`: **127,776**
- Доля от всех сессий: ~**6.87%**

📌 Интерпретация:
- Часть сессий не имеет ни одного зафиксированного события в `hits`.  
- Это допустимо и ожидаемо для веб-аналитики (обрывы, фильтрация событий, технические/короткие визиты и т.д.).

---

### 4) Дубли по ключу (session_id, hit_number)
Проверка уникальности пары `(session_id, hit_number)`:
- Количество “дублирующихся пар”: **225,862**
- Всего уникальных пар `(session_id, hit_number)`: **15,500,608**
- Доля дублирующихся пар: **1.4571%**
- Максимум строк на одну пару: **2**

Дополнительная диагностика дубликатов (пример):
- Во всех просмотренных дублях наблюдается типичный паттерн: **ровно одна строка из пары** имеет `NULL` в полях:
  - `hit_time` (1 из 2)
  - `hit_referer` (1 из 2)
  - `event_label` (1 из 2)

📌 Интерпретация:
- `(session_id, hit_number)` **нельзя использовать как первичный ключ** события.
- Дубли ограничены (строго до 2 строк), то есть это “контролируемая” особенность данных, а не ошибка загрузки.

---

### 5) Правила использования БД в проекте (принятые решения)
1. **Primary key для событий**: используем `hits.hit_id` (surrogate key), а не `(session_id, hit_number)`.
2. **Orphan hits**:
   - По умолчанию оставляем в базе как часть исходных данных.
   - При подготовке обучающего датасета можно:
     - либо исключать их при JOIN (inner join по `session_id`),
     - либо учитывать отдельно как “incomplete sessions” (если потребуется).
3. **Sessions without hits**:
   - Для задач конверсии такие сессии трактуем как “без события” (конверсия = 0), если таргет строится по `hits`.
4. **Дедупликация**:
   - На уровне базы **не выполняем** дедуп без явного правила.
   - Если на следующем этапе потребуется “1 hit на hit_number”, вводим формальное правило отбора (например, предпочитать строку с заполненным `hit_time`) и фиксируем его в ETL.

---

### 6) Итог
- База данных создана и заполнена корректно, по ключевым метрикам загрузки и датам — консистентна.
- Обнаруженные аномалии (сироты и дубли) имеют небольшой масштаб и объяснимую природу.
- Определены устойчивые правила работы с данными для последующих этапов (feature engineering / моделирование).

✅ Этап 3.2 завершён: SQLite готова как единый источник данных для дальнейшего пайплайна.
